In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control

from scipy.stats import chi2
from snp_analysis_tools_sherlock import *
from scipy.stats import binom
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
x=19
n=41
exp_val_unbiased = n/2
2*x*np.log(x/exp_val_unbiased) + 2*(n-x)*np.log((n-x)/exp_val_unbiased)

In [ ]:
1-chi2.cdf(2*x*np.log(x/exp_val_unbiased) + 2*(n-x)*np.log((n-x)/exp_val_unbiased), 1) 

In [ ]:
def get_Gstat(x,n):

    exp_val_unbiased = n/2
 #   if x== 0:
  #      return 2*n*np.log(2)
  #  if x == n:
   #     return 2*n*np.log(2)
    Gstat = 2*x*np.log(x/exp_val_unbiased) + 2*(n-x)*np.log((n-x)/exp_val_unbiased)
    return Gstat

def get_bias(df, subjects, groupby = 'mesocosm'):
    df['counts'] = 1
    df[f'winner {subjects[0]}'] = df[f'winner {subjects[0]}']>0
    df[f'winner {subjects[1]}'] = df[f'winner {subjects[1]}']>0
    dfgr = df.groupby(groupby).sum().reset_index()
    dfgr['subject1_counts'] = dfgr[f'winner {subjects[0]}']
   # dfgr['Gstat'] = -1 
    
    Gstats = np.zeros(len(dfgr))
    ns = dfgr['counts'].values
    xs = dfgr[f'winner {subjects[0]}'].values
    for i in range(len(dfgr)):
        x = xs[i]
        n = ns[i]
        Gstats[i] = get_Gstat(x,n)
      #  print(x,n)
        
    dfgr['Gstat'] = Gstats
    return  dfgr
    

def get_bias(df, subjects, groupby = 'mesocosm'):
    df['counts'] = 1
    df[f'winner {subjects[0]}'] = df[f'winner {subjects[0]}']>0
    df[f'winner {subjects[1]}'] = df[f'winner {subjects[1]}']>0
    dfgr = df.groupby(groupby).sum().reset_index()
    dfgr['subject1_counts'] = dfgr[f'winner {subjects[0]}']
   # dfgr['Gstat'] = -1 
    
    Gstats = np.zeros(len(dfgr))
    ns = dfgr['counts'].values
    xs = dfgr[f'winner {subjects[0]}'].values
    for i in range(len(dfgr)):
        x = xs[i]
        n = ns[i]
        Gstats[i] = get_Gstat(x,n)
      #  print(x,n)
        
    dfgr['Gstat'] = Gstats
    return  dfgr


    
    
    

In [ ]:
def calculate_qvalues(pvalues):

    # Uses the formula
    #
    # Qi = min_{Q>Pi} { Q*(sum_j 1)/(sum_j theta(Q-Pj)) } 
    #
    # Qi = max_{P_(k)>Pi} 

    # calculate q-values
    qvalues = []
    
    sorted_pvalues = np.array(sorted(pvalues))

    Ntot = len(sorted_pvalues)
    Nless =np.array([(sorted_pvalues<=p).sum() for p in sorted_pvalues])

    for p in pvalues:

        min_q = 1e06
            
        for j in reversed(range(0,Ntot)):
             
            if sorted_pvalues[j]<p:
                break
            else:
                new_q = Ntot*sorted_pvalues[j]*1.0/Nless[j]
                if new_q<min_q:
                    min_q = new_q
                    
        qvalues.append(min_q)
    
    qvalues = np.array(qvalues)
    
    return qvalues

In [ ]:
full_dfp7 = pd.read_csv('selection_coefficients_fitting2_v2.csv').set_index('species-mesocosm')

full_dfp7 = full_dfp7.loc[~full_dfp7['parent_subjects'].isin(['AA-AC/PP','AC/PP-AE','AC/PP-AF']),:]

full_dfp7['parent_subjects'].unique()
full_dfp7['type_mesocosm'] = full_dfp7['parent_subjects'] + '-' + full_dfp7['parent_media'] \
    + '-' + full_dfp7['media']
full_dfp7.loc[full_dfp7['p7_s'].isna(),'p7_s'] = full_dfp7.loc[full_dfp7['p7_s'].isna(),'p5_s']
full_dfp7 = full_dfp7.loc[~full_dfp7['p7_s'].isna(),:]
full_dfp7['parent1'] = full_dfp7['parent_subjects'].transform(lambda x: x.split('-')[0])
full_dfp7['winner']='parent1'
full_dfp7['parent2'] = full_dfp7['parent_subjects'].transform(lambda x: x.split('-')[1])
full_dfp7['loser']='parent2'
full_dfp7.loc[full_dfp7['p7_s']>0,'winner'] = full_dfp7.loc[full_dfp7['p7_s']>0,'parent1'] 
full_dfp7['winner_numeric'] = (full_dfp7['p7_s']>0).astype(int)
full_dfp7.loc[full_dfp7['p7_s']<0,'winner'] = full_dfp7.loc[full_dfp7['p7_s']<0,'parent2'] 
full_dfp7.loc[full_dfp7['p7_s']>0,'loser'] = full_dfp7.loc[full_dfp7['p7_s']>0,'parent2'] 
full_dfp7.loc[full_dfp7['p7_s']<0,'loser'] = full_dfp7.loc[full_dfp7['p7_s']<0,'parent1'] 
full_dfp7['counts']=1
from scipy.spatial.distance import jensenshannon
df_abundance = pd.read_csv('coal_sp_abundance.csv').drop(columns='Unnamed: 0')
goodsp = df_abundance.groupby(['species_id']).max().reset_index()
goodsp = goodsp.loc[goodsp['relative_abundance']>0,'species_id'].values
df_abundance = df_abundance.loc[df_abundance['species_id'].isin(goodsp),:]
df_abundance['mesocosm-passage'] = df_abundance['mesocosm'] + '-' + df_abundance['passage'].astype(str)

df_abundance['relative_abundance'].min()

metadata = pd.read_csv('e003_metadata_cultures_round2.csv')
samples= []
full_dfp7['dist_parent1'] = 0
full_dfp7['dist_parent2'] = 0
full_dfp7['dist_parent_inoculumn'] = 0
for sample in df_abundance['sample'].unique():
    spog_df = df_abundance.loc[df_abundance['sample'] == sample,:].sort_values(by='species_id')
    mesocosm = spog_df['type_mesocosm'].unique()[0]
    parent_subject1, parent_subject2, parent_media, media = mesocosm.split('-')
    
    ins = metadata.loc[metadata['is_inoculumn'],:]
    ins = ins.loc[ins['parent_media'] == parent_media,:]
    ins1 = ins.loc[ins['parent_subjects'] == parent_subject1 + '-' +  parent_subject1,'sample'].values[0]
    ins2 = ins.loc[ins['parent_subjects'] == parent_subject2 + '-' +  parent_subject2,'sample'].values[0]
    ins3 = ins.loc[ins['parent_subjects'] == parent_subject1 + '-' +  parent_subject2,'sample'].values[0]
    
    if ins1 == 'A2-e003Coalescence-Inoculumn-mBHI':
        ins1 = 'A2-e003Coalescence-mBHI-inoculumn-redo'
    print(ins1,ins2,ins3)
    sp1 = df_abundance.loc[df_abundance['sample'] == ins1,:].sort_values(by='species_id')
    sp2 = df_abundance.loc[df_abundance['sample'] == ins2,:].sort_values(by='species_id')
    sp3 = df_abundance.loc[df_abundance['sample'] == ins3,:].sort_values(by='species_id')

    JSD1 = jensenshannon(spog_df['relative_abundance'].values,sp1['relative_abundance'].values)
    JSD2 = jensenshannon(spog_df['relative_abundance'].values,sp2['relative_abundance'].values)
    JSD3 = jensenshannon(spog_df['relative_abundance'].values,sp3['relative_abundance'].values)
    samples.append(sample)
  #  dist_parent1.append(JSD1)
   # dist_parent2.append(JSD2)
   # dist_inoculumn.append(JSD3) 

    full_dfp7.loc[full_dfp7['sample'] == sample,'dist_parent1'] = JSD1
    full_dfp7.loc[full_dfp7['sample'] == sample,'dist_parent2'] = JSD2
    full_dfp7.loc[full_dfp7['sample'] == sample,'dist_parent_inoculumn'] = JSD3

In [ ]:

def boot_gstats(gstats, bootstraps = 1000):
    meds = np.zeros(bootstraps)
    for b in range(bootstraps):
       # print(b)
        new_gstats= np.random.choice(gstats, size = len(gstats))
        meds[b] = np.median(new_gstats)
    return meds


In [ ]:
full_dfp7_gr = full_dfp7.groupby(['type_mesocosm','species_id','parent_subjects','media','inoculumn',]).median(numeric_only=True).reset_index()
full_dfp7_gr['winner_numeric'] = full_dfp7_gr['p7_s']>0.
full_dfp7_gr['counts']=1
full_dfp7_gr['het'] = full_dfp7_gr['winner_numeric']*(full_dfp7_gr['winner_numeric']-full_dfp7_gr['counts'])
#full_dfp7_gr_Good = full_dfp7_gr.loc[full_dfp7_gr['het']==0,:]
full_dfp7_gr_gr = full_dfp7_gr.groupby(['type_mesocosm']).sum(numeric_only=True).reset_index()
full_dfp7_gr_gr['fraction_one_winner'] = full_dfp7_gr_gr['winner_numeric']/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['Gstat'] =  get_Gstat(full_dfp7_gr_gr['winner_numeric'],full_dfp7_gr_gr['counts'])
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == 0,'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == full_dfp7_gr_gr['counts'],'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr['chi_cdf'] = chi2.cdf(full_dfp7_gr_gr['Gstat'], 1) 
full_dfp7_gr_gr['invchi_cdf'] = 1 - full_dfp7_gr_gr['chi_cdf'] 
#full_dfp7_gr_gr_Good = full_dfp7_gr_gr_Good.loc[full_dfp7_gr_gr_Good['counts']>4,:]
pvals = false_discovery_control(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values)
#print(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'])
#pvals
print(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values)
print(pvals)
print(calculate_qvalues(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values))
full_dfp7_gr['full_counts'] =  full_dfp7_gr['type_mesocosm'].transform(lambda x: \
                                            full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'] == x,'counts'].values[0])
full_dfp7_gr['relative_counts'] = full_dfp7_gr['counts']/full_dfp7_gr['full_counts'] 

full_dfp7_grplot = full_dfp7_gr.reset_index()
#full_dfp7_grplot = full_dfp7_grplot.loc[full_dfp7_grplot['full_counts']>4,:]
full_dfp7_grplot1 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 1,:].copy()
full_dfp7_grplot1['parent'] =full_dfp7_grplot1['parent_subjects'].transform(lambda x: x.split('-')[0]) 
full_dfp7_grplot2 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 0,:].copy()
full_dfp7_grplot2['parent'] =full_dfp7_grplot2['parent_subjects'].transform(lambda x: x.split('-')[1]) 
not_sig_mesos= full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'].values[pvals>.05]

full_dfp7_gr_gr_not_sig = full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'].isin(not_sig_mesos),:]
#print(len(not_sig_mesos))
#print(len(full_dfp7_gr_gr))
new_gstats = boot_gstats(full_dfp7_gr_gr_not_sig['Gstat'].values)
print(np.percentile(new_gstats,97.5))
print(np.percentile(new_gstats,2.5))
print(np.median(full_dfp7_gr_gr_not_sig['Gstat'].values), np.percentile(full_dfp7_gr_gr_not_sig['Gstat'].values, 90))
full_dfp7_grplotfull = pd.concat([full_dfp7_grplot1, full_dfp7_grplot2]).sort_values(by='type_mesocosm',ascending=False)


bars = hv.Bars(full_dfp7_grplotfull , kdims=['type_mesocosm','parent'],
               vdims = ['relative_counts'])

bars.opts(width=600,height = 400, ).opts(stacked=True, ylabel='Fraction winner', #xlabel='
                                         invert_axes=True,#xrotation = 90,
                                         cmap = bokeh.palettes.Set3[12])

bars.opts(legend_position='left')#4*6

In [ ]:
full_dfp7_grplotfull.loc[full_dfp7_grplotfull['inoculumn'] == 'AE-AF-mGAM',['media','species_id','p7_s']].sort_values(by='species_id')

In [ ]:
full_dfp7_gr = full_dfp7.groupby(['type_mesocosm','species_id','parent_subjects','media',]).median(numeric_only=True).reset_index()
full_dfp7_gr['het'] = full_dfp7_gr['winner_numeric']*(full_dfp7_gr['winner_numeric']-full_dfp7_gr['counts'])
full_dfp7_gr_Good = full_dfp7_gr.loc[full_dfp7_gr['het']==0,:].copy()
full_dfp7_gr_Good['winner_numeric'] = full_dfp7_gr_Good['p7_s']>0.
full_dfp7_gr_Good['counts']=1
full_dfp7_gr_gr = full_dfp7_gr_Good.groupby(['type_mesocosm']).sum(numeric_only=True).reset_index()
full_dfp7_gr_gr['fraction_one_winner'] = full_dfp7_gr_gr['winner_numeric']/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['Gstat'] =  get_Gstat(full_dfp7_gr_gr['winner_numeric'],full_dfp7_gr_gr['counts'])
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == 0,'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == full_dfp7_gr_gr['counts'],'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr['chi_cdf'] = chi2.cdf(full_dfp7_gr_gr['Gstat'], 1) 
full_dfp7_gr_gr['invchi_cdf'] = 1 - full_dfp7_gr_gr['chi_cdf'] 
#full_dfp7_gr_gr_Good = full_dfp7_gr_gr_Good.loc[full_dfp7_gr_gr_Good['counts']>4,:]
pvals = false_discovery_control(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values)
pvals
print(pvals)

print(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'].values)
not_sig_mesos= full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'].values[pvals>.05]

full_dfp7_gr_gr_not_sig = full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'].isin(not_sig_mesos),:]
new_gstats = boot_gstats(full_dfp7_gr_gr_not_sig['Gstat'].values)
print(np.median(full_dfp7_gr_gr_not_sig['Gstat']))
print(np.percentile(new_gstats,97.5))
print(np.percentile(new_gstats,2.5))
    
full_dfp7_gr_Good['full_counts'] =  full_dfp7_gr_Good['type_mesocosm'].transform(lambda x: \
                                            full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'] == x,'counts'].values[0])
full_dfp7_gr_Good['relative_counts'] = full_dfp7_gr_Good['counts']/full_dfp7_gr_Good['full_counts'] 

full_dfp7_grplot = full_dfp7_gr_Good.reset_index()
#full_dfp7_grplot = full_dfp7_grplot.loc[full_dfp7_grplot['full_counts']>4,:]
full_dfp7_grplot1 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 1,:].copy()
full_dfp7_grplot1['parent'] =full_dfp7_grplot1['parent_subjects'].transform(lambda x: x.split('-')[0]) 
full_dfp7_grplot2 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 0,:].copy()
full_dfp7_grplot2['parent'] =full_dfp7_grplot2['parent_subjects'].transform(lambda x: x.split('-')[1]) 


full_dfp7_grplotfull = pd.concat([full_dfp7_grplot1, full_dfp7_grplot2]).sort_values(by='type_mesocosm',ascending=False)


bars = hv.Bars(full_dfp7_grplotfull , kdims=['type_mesocosm','parent'],
               vdims = ['relative_counts'])

bars.opts(width=600,height = 400, ).opts(stacked=True, ylabel='Fraction winner', #xlabel='
                                         invert_axes=True,#xrotation = 90,
                                         xlabel = 'Competitions won by subject x'
                                         cmap = bokeh.palettes.Set3[12])

bars.opts(legend_position='left')#4*6

In [ ]:
from scipy.stats import binom
binom.pmf(0,9,p=.5)

In [ ]:
full_dfp7_gr = full_dfp7.groupby(['mesocosm','type_mesocosm','species_id','parent_subjects','media',]).median(numeric_only=True).reset_index()

full_dfp7_gr['winner_numeric'] = full_dfp7_gr['p7_s']>0.

full_dfp7_gr['counts']=1
full_dfp7_gr_gr = full_dfp7_gr.groupby(['type_mesocosm','media','mesocosm']).sum(numeric_only=True)

full_dfp7_gr_gr['fraction_one_winner'] = full_dfp7_gr_gr['winner_numeric']/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['dist_parent1'] = full_dfp7_gr_gr['dist_parent1']/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['dist_parent2']= full_dfp7_gr_gr['dist_parent2']/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['dist_parent_inoculumn']= full_dfp7_gr['dist_parent_inoculumn']/full_dfp7_gr['counts']
full_dfp7_gr_gr[['fraction_one_winner','dist_parent1','dist_parent2']]
full_dfp7_gr_gr['dist_dist'] = full_dfp7_gr_gr['dist_parent1']-full_dfp7_gr_gr['dist_parent2']
full_dfp7_gr_gr['p7_s'] = full_dfp7_gr_gr['p7_s']/full_dfp7_gr_gr['counts']
dfplot = full_dfp7_gr_gr#.loc[full_dfp7_gr_gr['counts']>5,:].copy() 

dfplot['dominant'] = dfplot['fraction_one_winner']>.5
dfplot.loc[dfplot['dominant'],'fraction_one_winner'] = 1-dfplot.loc[dfplot['dominant'],'fraction_one_winner'] 
dfplot.loc[dfplot['dominant'],'p7_s'] = -dfplot.loc[dfplot['dominant'],'p7_s'] 
dfplot['dist_subdominant'] = dfplot['dist_parent1'] 
dfplot.loc[dfplot['dominant'], 'dist_subdominant'] = dfplot['dist_parent2'] 
dfplot['dominant-subdom'] = -dfplot['dist_dist']
dfplot.loc[dfplot['dominant'],'dominant-subdom'] = dfplot['dist_dist']

#.loc[full_dfp7_gr_gr['counts']>4,:]
#full_dfp7_gr_gr = full_dfp7_gr.groupby(['type_mesocosm']).median(numeric_only=True)
#hv.Scatter(data = dfplot.reset_index(), kdims = ['dist_dist'],vdims = ['fraction_one_winner', 'parent_subjects'])

dfplot = dfplot.reset_index()
scatter1 = hv.Scatter(dfplot, kdims = ['dominant-subdom'],vdims = ['fraction_one_winner',
                                                                           hv.Dimension('type_mesocosm')]).opts(size=6, 
                                                                                 color = 'type_mesocosm',
                                                                                       width = 800,#height=600,
                                                                                                                cmap=bokeh.palettes.Category20[15],
                                                                                                 line_color='black',
                                                                                         colorbar=True, 
                                                                                        legend_position='right',
                                                                                       #  title = 'Leave One out Frequency Pred',
                                                                                                  #alpha = .8,
                                                                                             #xlim=(0.001,1.001),ylim=(0.001,1.001),
                                                                                               xlabel='JSD dominant - JSD subdominant',
                                                                                             # legend_label = 'JSD_dist',
                                                                                              
                                                                                               ylabel='Fraction Subdominant')#,ylim=(-0.05,.55))

                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

scatter1

In [ ]:
from scipy.stats import pearsonr


pearsonr(dfplot.loc[dfplot['media'] == 'mBHI','fraction_one_winner'].values,dfplot.loc[dfplot['media'] == 'mBHI','dominant-subdom'])

In [ ]:
def boot_strap_JSD(fractions, JSDs, bootstraps = 1000):
    rs = np.zeros(bootstraps)
    for b in range(bootstraps):
       # print(b)
        new_JSDs = np.copy(JSDs)
        new_dirs = np.random.choice([1,-1], size=len(new_JSDs))
        new_JSDs = new_JSDs*new_dirs
        new_corr_coeff = pearsonr(fractions, new_JSDs)[0]
        rs[b] = new_corr_coeff
    return rs
        

In [ ]:
np.log10(100000)

In [ ]:
pearsonr(dfplot['fraction_one_winner'],dfplot['dominant-subdom'])[0]

In [ ]:
pearsonr(dfplot.loc[dfplot['media'] == 'mBHI','fraction_one_winner'].values,dfplot.loc[dfplot['media'] == 'mBHI','dominant-subdom'])
rs = boot_strap_JSD(dfplot.loc[dfplot['media'] == 'mBHI','fraction_one_winner'].values,dfplot.loc[dfplot['media'] == 'mBHI','dominant-subdom'], bootstraps = 100000)
#rs = boot_strap_JSD(b,c,bootstraps=1000)
print(np.percentile(rs, 97.5), np.percentile(rs, 2.5))

In [ ]:
b['p7_s'].values,b['dist_dist'].values

In [ ]:
p = iqplot.ecdf(rs)
p.ray(x = pearsonr(dfplot['fraction_one_winner'],dfplot['dominant-subdom'])[0],y=0,
      color = 'red', angle=np.pi/2)
bokeh.io.show(p)

In [ ]:
full_dfp7_gr['counts']=1.
full_dfp7_gr = full_dfp7.groupby(['type_mesocosm','species_id','parent_subjects','media',]).median(numeric_only=True).reset_index()
full_dfp7_gr['p7_s'] = full_dfp7_gr['p7_s']#/full_dfp7_gr['counts']
full_dfp7_gr_gr['winner_numeric'] = full_dfp7_gr['p7_s']>0.

full_dfp7_gr_gr['counts']=1
full_dfp7_gr_gr = full_dfp7_gr.groupby(['type_mesocosm']).median(numeric_only=True)

full_dfp7_gr_gr['fraction_one_winner'] = full_dfp7_gr_gr['winner_numeric']#/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['dist_parent1'] = full_dfp7_gr_gr['dist_parent1']#/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['dist_parent2']= full_dfp7_gr_gr['dist_parent2']#/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['dist_parent_inoculumn']= full_dfp7_gr['dist_parent_inoculumn']#/full_dfp7_gr['counts']
full_dfp7_gr_gr[['fraction_one_winner','dist_parent1','dist_parent2']]
full_dfp7_gr_gr['dist_dist'] = full_dfp7_gr_gr['dist_parent1']-full_dfp7_gr_gr['dist_parent2']
full_dfp7_gr_gr['p7_s'] =full_dfp7_gr_gr['p7_s']#/full_dfp7_gr_gr['counts']
dfplot = full_dfp7_gr_gr#loc[full_dfp7_gr_gr['counts']>5,:].copy() 

dfplot['dominant'] = dfplot['p7_s']>.0
dfplot.loc[dfplot['dominant'],'fraction_one_winner'] = 1-dfplot.loc[dfplot['dominant'],'fraction_one_winner'] 
dfplot.loc[dfplot['dominant'],'p7_s'] = -dfplot.loc[dfplot['dominant'],'p7_s'] 
dfplot['dist_subdominant'] = dfplot['dist_parent1'] 
dfplot.loc[dfplot['dominant'], 'dist_subdominant'] = dfplot['dist_parent2'] 
dfplot['dominant-subdom'] = -dfplot['dist_dist']
dfplot.loc[dfplot['dominant'],'dominant-subdom'] = dfplot['dist_dist']

#.loc[full_dfp7_gr_gr['counts']>4,:]
#full_dfp7_gr_gr = full_dfp7_gr.groupby(['type_mesocosm']).median(numeric_only=True)
#hv.Scatter(data = dfplot.reset_index(), kdims = ['dist_dist'],vdims = ['fraction_one_winner', 'parent_subjects'])


scatter1 = hv.Scatter(dfplot.reset_index(), kdims = ['dominant-subdom'],vdims = ['p7_s',
                                                                           hv.Dimension('type_mesocosm')]).opts(size=6, 
                                                                                 color = 'type_mesocosm',
                                                                                       width = 800,#height=600,
                                                                                        cmap=bokeh.palettes.Set2[8],
                                                                                                 line_color='black',
                                                                                         colorbar=True, 
                                                                                        legend_position='right',
                                                                                       #  title = 'Leave One out Frequency Pred',
                                                                                                  #alpha = .8,
                                                                                             #xlim=(0.001,1.001),ylim=(0.001,1.001),
                                                                                               xlabel='JSD dominant - JSD subdominant',
                                                                                             # legend_label = 'JSD_dist',
                                                                                              
                                                                                               ylabel='Avg Sel Coefficient',)#ylim=(-0.05,.55))

                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

scatter1

In [ ]:
from scipy.stats import pearsonr


dfplot = dfplot.loc[dfplot['p7_s']>-1.,:]
rs = boot_strap_JSD(dfplot['fraction_one_winner'].values,dfplot['p7_s'].values, bootstraps = 100000)
print(pearsonr(dfplot['fraction_one_winner'],dfplot['p7_s']))
#rs = boot_strap_JSD(b,c,bootstraps=1000)
print(np.percentile(rs, 97.5), np.percentile(rs, 2.5))